In [19]:
import pandas as pd
import warnings
import numpy as np
from typing import Literal
import chardet
import os
warnings.filterwarnings('ignore')

In [20]:
def find_encoding(file_path):
    with open(file_path, 'rb') as file:
        raw_data = file.read()
        detected = chardet.detect(raw_data)
        encoding = detected['encoding']

    print(f"Encodage détecté : {encoding}")
    

In [21]:
# import os

# # Get current working directory
# cwd = os.getcwd()

# # Define the folder you want to check against
# target_folder = 'Revue Analytique'

# # Check if current directory ends with the target folder name
# if cwd.endswith(target_folder):
#     print(f"You're already in the correct folder: {cwd}")
# else:
#     os.chdir('..')
#     print(f"Moved up one directory. Now in: {os.getcwd()}")


In [22]:
file_path = r"FECS\KANGOUROU\394982995FEC20231231.txt"
# find_encoding(file_path)

In [23]:
liste_encoding = ["utf-8","UTF-16","MacRoman","SHIFT_JIS","CP932"]

i = 0
success = False
while i < len(liste_encoding) and not success:
    try:
        df = pd.read_csv(file_path, sep='\t', encoding=liste_encoding[i], engine='python')
        success = True
    except (UnicodeDecodeError,UnicodeError):
        i += 1

annee = 2025
df.head()

,JournalCode,JournalLib,EcritureNum,EcritureDate,CompteNum,CompteLib,CompAuxNum,CompAuxLib,PieceRef,PieceDate,EcritureLib,Debit,Credit,EcritureLet,DateLet,ValidDate,Montantdevise,Idevise
0,AN,A NOUVEAUX,AN000001,20230101,44566000,TVA déductible,NaN,NaN,OD,20230101,EXT REAFFECTATION,"0,00","7,00",NaN,NaN,20231231,NaN,NaN
1,AN,A NOUVEAUX,AN000001,20230101,44566000,TVA déductible,NaN,NaN,1,20230101,PRLV GRENKE LOCATION,"112,20","0,00",NaN,NaN,20231231,NaN,NaN
2,AN,A NOUVEAUX,AN000001,20230101,44566000,TVA déductible,NaN,NaN,19010001,20230101,CB CARTE X4951 10/01 STATIOMT0,"2,50","0,00",NaN,NaN,20231231,NaN,NaN
3,AN,A NOUVEAUX,AN000001,20230101,44566000,TVA déductible,NaN,NaN,19010002,20230101,CB CARTE X4951 10/01 BP BOUGIV,"5,67","0,00",NaN,NaN,20231231,NaN,NaN
4,AN,A NOUVEAUX,AN000001,20230101,44566000,TVA déductible,NaN,NaN,19010008,20230101,000001 VIR EUROPEEN EMIS NET,"997,29","0,00",NaN,NaN,20231231,NaN,NaN


In [24]:
df.columns = df.columns.str.strip()
df["EcritureDate"] = pd.to_datetime(df["EcritureDate"],format='%Y%m%d', errors='coerce')
df['Année'] = df["EcritureDate"].dt.year
df['Mois'] = df["EcritureDate"].dt.month_name(locale='French')

def reorder_months(df):
    mois_en_ordre = ["Janvier", "Février", "Mars", "Avril", "Mai", "Juin", "Juillet", "Août", "Septembre", "Octobre", "Novembre", "Décembre"]
    # Get all unique years in ascending order
    years = sorted(df['Année'].unique())
    
    if len(years) == 0:
        return mois_en_ordre   # Return original order if no data found
    
    # Get the first month for the oldest year
    oldest_year = years[0]
    year_data = df[df['Année'] == oldest_year]
    # Convert Mois to categorical with the correct order
    year_data['Mois'] = pd.Categorical(year_data['Mois'], 
                                     categories=mois_en_ordre,
                                     ordered=True)
    
    # Now min() will respect our month order
    first_month = year_data['Mois'].min()
 
    
    if pd.isna(first_month):
        return mois_en_ordre  # Return original order if no valid month found
    
    # Find the index of the first month in the original list
    start_idx = mois_en_ordre.index(first_month)
    
    # Create new list starting from first_month
    reordered_months = mois_en_ordre[start_idx:] + mois_en_ordre[:start_idx]
    
    return reordered_months

# Replace the original mois_en_ordre assignment with:
mois_en_ordre = reorder_months(df)

df["Credit"] = pd.to_numeric(df["Credit"].str.replace(',', '.', regex=True), errors='coerce').fillna(0)
df["Debit"] = pd.to_numeric(df["Debit"].str.replace(',', '.', regex=True), errors='coerce').fillna(0)

def custom_round(value):
    # If the decimal part is greater than or equal to 0.5, round up
    if value - np.floor(value) >= 0.5:
        return int(np.ceil(value))  # Round up and convert to int
    else:
        return int(np.floor(value))  # Round down and convert to int


In [25]:
def calculate_custom_cumsum_old(df, columns=None, add_total=False, complete_remaining=False):
    """
    Custom cumulative sum:
    - Starts summing from the first nonzero value
    - Includes intermediate zeros and NaNs in the running total
    - When a NaN is encountered, it uses the last cumulative value
    - When a zero is encountered, it maintains the running total
    - If complete_remaining=True:
      - Replaces all trailing NaNs with the last cumulative total
      - Replaces all trailing zeros with the last cumulative total
    - If complete_remaining=False:
      - Only non-trailing zeros and NaNs maintain the running total
    """
    result_df = df.copy()
    
    if columns is None:
        columns = df.columns[1:]
        
    last_values = {}
    
    for column in columns:
        # Get column values as numpy array for efficient processing
        values = result_df[column].values
        n = len(values)
        
        # Initialize result array with zeros
        result = np.zeros(n)
        
        # Find first non-zero position
        first_nonzero = n
        for i in range(n):
            if not pd.isna(values[i]) and values[i] != 0:
                first_nonzero = i
                break
        
        # If no non-zero values found, continue to next column
        if first_nonzero == n:
            result_df[column] = result
            continue
            
        # Calculate cumsum starting from first non-zero
        cumulative_total = 0
        last_nonzero_position = first_nonzero
        
        for i in range(first_nonzero, n):
            if pd.isna(values[i]) or values[i] == 0:
                # For zeros and NaNs, keep the cumulative total
                result[i] = cumulative_total
            else:
                # For non-zero values, add to cumulative total
                cumulative_total += values[i]
                result[i] = cumulative_total
                last_nonzero_position = i  # Track position of last non-zero value
        
        # Store the final cumulative total
        last_values[column] = cumulative_total if cumulative_total != 0 else None
        
        # Get the cumulative total before the trailing zeroes/NaNs
        final_cumulative_total = result[last_nonzero_position]
        
        # If complete_remaining is False, reset trailing zeros and NaNs to 0
        # This effectively undoes what we set earlier for trailing zeros/NaNs
        if not complete_remaining and last_nonzero_position < n - 1:
            for i in range(last_nonzero_position + 1, n):
                result[i] = 0
        
        # Only if complete_remaining is True, handle trailing zeros and NaNs
        if complete_remaining and cumulative_total != 0 and last_nonzero_position < n - 1:
            for i in range(last_nonzero_position + 1, n):
                if pd.isna(values[i]) or values[i] == 0:
                    result[i] = final_cumulative_total
        
        # Set the result in the DataFrame
        result_df[column] = result
        result_df[column] = result_df[column].apply(custom_round)

    if add_total:
        total_row = ['Total']
        for column in columns:
            # Use the last cumulative total for the total row
            total = last_values.get(column, 0)
            total_row.append(total if total is not None else 0)
        result_df.loc[len(result_df)] = total_row

    last_value = last_values[list(last_values.keys())[0]] if len(list(last_values.keys())) == 1 and len(last_values) > 0 else None

    return result_df, last_value

In [26]:

# Assuming custom_round is now fixed to handle NaNs gracefully
# For demonstration, let's define a safe custom_round that passes NaNs through
def custom_round(value):
    if pd.isna(value):
        return np.nan
    # Original custom_round logic
    if value - np.floor(value) >= 0.5:
        return int(np.ceil(value))
    else:
        return int(np.floor(value))

def calculate_custom_cumsum(df, columns=None, add_total=False):
    result_df = df.copy()
    
    if columns is None:
        columns = df.columns[1:]
        
    last_values = {}
    
    for column in columns:
        values = result_df[column].to_list()
        result = []
        is_nan_beginning = True
        last_cumulative_value = 0

        for value in values:
            if pd.isna(value):
                result.append(0 if is_nan_beginning else custom_round(last_cumulative_value))
            else:
                is_nan_beginning = False
                last_cumulative_value += value
                result.append(custom_round(last_cumulative_value))

        result_df[column] = result
        last_values[column] = last_cumulative_value

    if add_total:
        total_row = ['Total'] + [last_values.get(col, 0) for col in columns]
        result_df.loc[len(result_df)] = total_row

    last_value = last_values[list(last_values.keys())[0]] if len(last_values) == 1 else None

    return result_df, last_value

## CA

In [27]:
def chiffres_affaires(df):
    
    # Filtrer uniquement les comptes de chiffre d'affaires 
    ca = df[df['CompteNum'].astype(str).str[:3].astype(int).between(700, 709)]
    # Calculer les montants nets (Débit - Crédit)
    ca['Montant'] = ca["Credit"] - ca["Debit"]

    # Appliquer l'ordre des mois
    ca['Mois'] = pd.Categorical(ca['Mois'], categories=mois_en_ordre, ordered=True)

    if len(ca['Année'].unique()) == 1:
        # Grouper par année et mois et calculer le total
        ca_mensuel = ca.groupby(['Année','Mois'])['Montant'].sum().reset_index()
        ca_mensuel = ca_mensuel.pivot(index="Mois", columns="Année", values="Montant")
        ca_mensuel= ca_mensuel.map(custom_round)
        ca_mensuel.columns.name = None
        ca_mensuel = ca_mensuel.reset_index()
    else:
        ca_mensuel = ca.groupby(['Mois'])['Montant'].sum().reset_index()
        ca_mensuel['Montant']= ca_mensuel['Montant'].map(custom_round)
        ca_mensuel.rename(columns={'Montant': '-'.join([str(annee) for annee in ca['Année'].unique()])}, inplace=True)
    
    colonne_annee = list(ca_mensuel.columns)[1]
    # On calcule les chiffres d'affaires cumulés
    ca_mensuel_cumule,total_cumule = calculate_custom_cumsum(ca_mensuel)
    ca_mensuel.loc[len(ca_mensuel)] = ['Total', ca_mensuel[colonne_annee].sum()]

    ca_mensuel["Pourcentage du CA Total"] = ca_mensuel.apply(lambda row: f'{round((row[colonne_annee]/ca_mensuel.loc[len(ca_mensuel)-1,colonne_annee])*100)}%',axis = 1)
    ca_mensuel.at[len(ca_mensuel)-1, "Pourcentage du CA Total"] = ""
    ca_mensuel.rename(columns={colonne_annee:str(colonne_annee)}, inplace=True)


    ca_mensuel_cumule["Pourcentage du CA Total"] = ca_mensuel_cumule.apply(lambda row: f'{round((row[colonne_annee]/total_cumule)*100)}%',axis = 1)
    ca_mensuel_cumule.rename(columns={colonne_annee:str(colonne_annee)}, inplace=True)
    return ca_mensuel ,ca_mensuel_cumule

In [28]:
ca_mensuel,ca_mensuel_cumule = chiffres_affaires(df)

In [29]:
ca_mensuel

,Mois,2023,Pourcentage du CA Total
0,Janvier,26454,3%
1,Février,52037,7%
2,Mars,98909,13%
3,Avril,83273,11%
4,Mai,39631,5%
5,Juin,207775,26%
6,Juillet,43127,5%
7,Août,24800,3%
8,Septembre,48220,6%
9,Octobre,31566,4%


In [30]:
ca_mensuel_cumule


,Mois,2023,Pourcentage du CA Total
0,Janvier,26454,3%
1,Février,78491,10%
2,Mars,177400,22%
3,Avril,260673,33%
4,Mai,300304,38%
5,Juin,508079,64%
6,Juillet,551206,70%
7,Août,576006,73%
8,Septembre,624226,79%
9,Octobre,655792,83%


## Trésorerie

### LES COMPTES COURANTS ET LES EMPRUNTS SONT CREDITEURS (SOLDE = CREDIT - DEBIT)

In [31]:
def tresorerie(df):
    """
    Génère un tableau de la trésorerie mensuelle 

    Parameters
    ----------
    df : pandas.DataFrame
        Le DataFrame contenant les données de la balance comptable
                    
    Returns
    -------
    pandas.DataFrame
        Un DataFrame contenant la trésorerie mensuelle brute, nette des emprunts, et nette des emprunts et des comptes courants
    """

    # Add this line at the start of the function
    mois_en_ordre = reorder_months(df)

    comptes_bancaires = df[df['CompteNum'].astype(str).str.startswith('5')]
    emprunts = df[df['CompteNum'].astype(str).str.startswith('16')]
    compte_courants = df[df['CompteNum'].astype(str).str.startswith('45')]

    def generer_table_tresorerie(data, type):
        # Retirer les lignes avec des dates invalides
        data = data.dropna(subset=["EcritureDate"])
        
        # Calcul du solde
        data["Solde"] = data["Debit"] - data["Credit"]    

        # Grouper par Mois et CompteLib en gardant tous les mois
        tresorerie_mensuelle = data.groupby(["CompteLib", "Mois"], observed=False)["Solde"].sum().reset_index()

        # Créer un DataFrame contenant tous les mois et comptes possibles
        all_combinations = pd.MultiIndex.from_product([data["CompteLib"].unique(), mois_en_ordre], names=["CompteLib", "Mois"])

        tresorerie_mensuelle = tresorerie_mensuelle.set_index(["CompteLib", "Mois"]).reindex(all_combinations).reset_index()

        # Pivot pour aligner les mois
        tresorerie_mensuelle_pivot = tresorerie_mensuelle.pivot(index="Mois", columns="CompteLib", values="Solde").reindex(mois_en_ordre).reset_index()
        tresorerie_mensuelle_pivot.columns.name = ""

        tresorerie_mensuelle_pivot,_ = calculate_custom_cumsum(tresorerie_mensuelle_pivot)
        

        # Calcul du total
        total_label = {
            "comptes bancaires": "Trésorerie Brute",
            "emprunts": "Total des emprunts",
        }.get(type, "Total des C/C")

        tresorerie_mensuelle_pivot[total_label] = tresorerie_mensuelle_pivot.iloc[:, 1:].sum(axis=1)



        return tresorerie_mensuelle_pivot
    
    
    tableau_comptes_bancaires = generer_table_tresorerie(
        comptes_bancaires,
        'comptes bancaires',
    )
    tableau_emprunts = generer_table_tresorerie(
        emprunts,
        'emprunts',
    )
    tableau_compte_courants = generer_table_tresorerie(
        compte_courants,
        'comptes courants',
    )
    

    tableau_emprunts["Trésorerie nette des emprunts"] = tableau_emprunts["Total des emprunts"] + tableau_comptes_bancaires["Trésorerie Brute"]

    tableau_compte_courants["Trésorerie nette des emprunts et CC"] = tableau_compte_courants["Total des C/C"] + tableau_emprunts["Trésorerie nette des emprunts"]

    
    tableau_tresorie = tableau_comptes_bancaires.merge(tableau_emprunts,on="Mois", how='outer').merge(tableau_compte_courants,on="Mois", how='outer')
    tableau_tresorie["Mois"] = pd.Categorical(
        tableau_tresorie["Mois"],
        categories=mois_en_ordre,
        ordered=True,
    )
    tableau_tresorie = tableau_tresorie.dropna(subset=['Mois'])
    tableau_tresorie = tableau_tresorie.sort_values("Mois").reset_index(drop=True)

    tableau_tresorie.loc[:, tableau_tresorie.columns!="Mois"] = tableau_tresorie.loc[:, tableau_tresorie.columns!="Mois"].fillna(0)
    numeric_columns = tableau_tresorie.select_dtypes(include=["number"]).columns
    tableau_tresorie[numeric_columns] = tableau_tresorie[numeric_columns].map(custom_round)
  
    tableau_tresorie["Trésorerie nette des emprunts"] = tableau_tresorie["Total des emprunts"] + tableau_tresorie["Trésorerie Brute"]
    tableau_tresorie["Trésorerie nette des emprunts et CC"] = tableau_tresorie["Total des C/C"] + tableau_tresorie["Trésorerie nette des emprunts"]


        
    return tableau_tresorie

In [32]:
data = df[df['CompteNum'].astype(str).str.startswith('45')]
data = data.dropna(subset=["EcritureDate"])

# Calcul du solde
data["Solde"] = data["Debit"] - data["Credit"]    

# Grouper par Mois et CompteLib en gardant tous les mois
tresorerie_mensuelle = data.groupby(["CompteLib", "Mois"], observed=False)["Solde"].sum().reset_index()

# Créer un DataFrame contenant tous les mois et comptes possibles
all_combinations = pd.MultiIndex.from_product([data["CompteLib"].unique(), mois_en_ordre], names=["CompteLib", "Mois"])

tresorerie_mensuelle = tresorerie_mensuelle.set_index(["CompteLib", "Mois"]).reindex(all_combinations).reset_index()

# Pivot pour aligner les mois
tresorerie_mensuelle_pivot = tresorerie_mensuelle.pivot(index="Mois", columns="CompteLib", values="Solde").reindex(mois_en_ordre).reset_index()
tresorerie_mensuelle_pivot.columns.name = ""

mois_present = tresorerie_mensuelle_pivot["Mois"].isin(data["Mois"].unique())
tresorerie_mensuelle_pivot,_ = calculate_custom_cumsum(tresorerie_mensuelle_pivot)
tresorerie_mensuelle_pivot

,Mois,Associes dividendes a payer,Principal
0,Janvier,0,-373
1,Février,0,-373
2,Mars,0,-444
3,Avril,0,-444
4,Mai,0,-444
5,Juin,0,-444
6,Juillet,0,-444
7,Août,15000,-444
8,Septembre,0,-444
9,Octobre,0,-444


In [33]:
tresorerie_resultat = tresorerie(df)
tresorerie_resultat

,Mois,Actions,Banque,Caisse,banque,Trésorerie Brute,Prêt,pret nø219252100344,Total des emprunts,Trésorerie nette des emprunts,Associes dividendes a payer,Principal,Total des C/C,Trésorerie nette des emprunts et CC
0,Janvier,5000,308,913,80694,86915,0,-319,-319,86596,0,-373,-373,86223
1,Février,5000,308,913,113906,120127,-20000,-41,-20041,100086,0,-373,-373,99713
2,Mars,5000,308,913,111806,118027,-19475,0,-19475,98552,0,-444,-444,98108
3,Avril,5000,308,913,122868,129089,-18949,0,-18949,110140,0,-444,-444,109696
4,Mai,5000,308,913,147258,153479,-18421,0,-18421,135058,0,-444,-444,134614
5,Juin,5000,308,913,104060,110281,-17891,0,-17891,92390,0,-444,-444,91946
6,Juillet,5000,308,913,112827,119048,-17360,0,-17360,101688,0,-444,-444,101244
7,Août,5000,308,913,116845,123066,-16827,0,-16827,106239,15000,-444,14556,120795
8,Septembre,5000,308,913,135044,141265,-16292,0,-16292,124973,0,-444,-444,124529
9,Octobre,5000,308,913,130915,137136,-15755,0,-15755,121381,0,-444,-444,120937


## Salaires

In [34]:
def frais_personnels(df):
    
    salaires = df[df['CompteNum'].astype(str).str.startswith('641')]
    charges_sociales = df[df['CompteNum'].astype(str).str.startswith('64') & ~df['CompteNum'].astype(str).str.startswith('641')]

    def generer_table_frais_personnels(data,type:Literal["Salaires","Charges sociales"]):

        data['Mois'] = pd.Categorical(data['Mois'], categories=mois_en_ordre, ordered=True)

        data[type] = data["Debit"] - data["Credit"]    

        frais_personnels = data.groupby(['Mois'])[type].sum().reset_index()
        frais_personnels[type] = frais_personnels[type].map(custom_round)
        frais_personnels.loc[len(frais_personnels)] = ['Total', frais_personnels[type].sum()]

        return frais_personnels

    table_salaire = generer_table_frais_personnels(salaires,
                                                   "Salaires")
    
    table_charges_sociales = generer_table_frais_personnels(charges_sociales,
                                                            "Charges sociales")
    
    table_frais_personnels = pd.merge(table_salaire,table_charges_sociales,on='Mois')

    table_frais_personnels["Total Salaires et Charges Sociales"] = table_frais_personnels["Salaires"] + table_frais_personnels["Charges sociales"]

    table_frais_personnels_cumules,__ = calculate_custom_cumsum(table_frais_personnels[["Mois", "Total Salaires et Charges Sociales"]][:-1])

    
    return table_frais_personnels,table_frais_personnels_cumules
    

In [35]:
frais_personnels,frais_personnels_cumules = frais_personnels(df)
frais_personnels

,Mois,Salaires,Charges sociales,Total Salaires et Charges Sociales
0,Janvier,2042,9859,11901
1,Février,6496,4653,11149
2,Mars,6330,7154,13484
3,Avril,6330,7353,13683
4,Mai,6672,10404,17076
5,Juin,8049,7775,15824
6,Juillet,11444,10203,21647
7,Août,6444,8139,14583
8,Septembre,7190,8194,15384
9,Octobre,6615,8142,14757


In [36]:
frais_personnels_cumules

,Mois,Total Salaires et Charges Sociales
0,Janvier,11901
1,Février,23050
2,Mars,36534
3,Avril,50217
4,Mai,67293
5,Juin,83117
6,Juillet,104764
7,Août,119347
8,Septembre,134731
9,Octobre,149488


## Crédit moyen client/ Crédit moyen fournisseurs

In [37]:
def generer_table_soldes(df,type,table_solde_data = None,index_column = "Mois"):
    """
    Génère un tableau des soldes mensuels pour les fournisseurs ou les clients
    pour une année donnée.

    Parameters
    ----------
    df : pandas.DataFrame
        Le DataFrame contenant les données de la balance comptable.
    type : Literal["Fournisseurs", "Clients"]
        Indique si le calcul des soldes doit être fait pour les "Fournisseurs" ou les "Clients".

    Returns
    -------
    pandas.DataFrame
        Un DataFrame avec les soldes mensuels cumulés pour chaque mois de l'année,
        avec une ligne 'Total' à la fin.
    """

    mois_en_ordre = reorder_months(df)

    # On définit le numéro du compte pour les fournisseurs ou les clients
    compte_num = '0'
    if type =="Fournisseurs":
        compte_num = '401'
    else:
        compte_num = '411'
    
    data = table_solde_data.loc[table_solde_data['CompteNum'].astype(str).str.startswith(compte_num)] if table_solde_data is not None else df.loc[df['CompteNum'].astype(str).str.startswith(compte_num)]

    # On trie les données par mois
    data["Mois"] = pd.Categorical(data[index_column], categories=mois_en_ordre, ordered=True)
    
    if type =="Fournisseurs":
        data["Soldes"] = data["Credit"] - data["Debit"]
    else:
        data["Soldes"] = data["Debit"] - data["Credit"]

    # On calcule les soldes mensuels cumulés
    calcul_soldes = data.groupby([index_column])["Soldes"].sum().reset_index()
    calcul_soldes["Soldes"] = calcul_soldes["Soldes"].map(custom_round)

    # Pour toutes les colonnes sauf la première:
    calcul_soldes, total_solde = calculate_custom_cumsum(calcul_soldes,add_total=True)

    return calcul_soldes, total_solde

In [38]:
def generer_table_jours(tableau_source, type, previous_year_data=None,index_column = "Mois"):
    """
    Génère un tableau montrant l'évolution du crédit moyen clients
    pour chaque mois de l'année spécifiée, avec une ligne 'Total' à la fin.

    Args:
        tableau_source: DataFrame contenant les données de la balance comptable
        type: ["Clients", "Fournisseurs"]
        previous_year_data: DataFrame contenant les données de l'année précédente

    Returns:
        data: Un DataFrame contenant le crédit moyen clients mensuel
    """
    colonne_ttc = "Total TTC" if type == "Clients" else "Achats TTC"
    jours = []
    data = tableau_source.copy()
    
    # Si les données de l'année précédente sont fournies, on les copie et on retire la dernière ligne
    if previous_year_data is not None:
        prev_data = previous_year_data.copy()
        prev_data = prev_data[[colonne_ttc]].iloc[:-1]
    else:
        prev_data = None
    
    for i in range(len(data)):
        solde = data['Soldes'].iloc[i]
        achat = data[colonne_ttc].iloc[i]
        
        if solde > achat:
            somme_achats_precedents = achat
            k = i
            
            while k > 0 and somme_achats_precedents < solde:
                k -= 1
                somme_achats_precedents += data[colonne_ttc].iloc[k]
            
            if somme_achats_precedents >= solde:
                somme_achats_precedents -= data[colonne_ttc].iloc[k]
                jours_valeur = ((solde - somme_achats_precedents) / data[colonne_ttc].iloc[k]) * 30 + 30 * (i - k)
            
            # Si on atteint la première ligne et que le solde n'est pas couvert, on passe à l'année précédente
            elif prev_data is not None:
                k_prev = len(prev_data) - 1
                while k_prev >= 0 and somme_achats_precedents < solde:
                    somme_achats_precedents += prev_data[colonne_ttc].iloc[k_prev]
                    k_prev -= 1
                
                if somme_achats_precedents >= solde:
                    somme_achats_precedents -= prev_data[colonne_ttc].iloc[k_prev + 1]
                    jours_valeur = ((solde - somme_achats_precedents) / prev_data[colonne_ttc].iloc[k_prev + 1]) * 30 + 30 * (len(prev_data) - k_prev - 1 + i)
                else:
                    jours_valeur = 0
            else:
                jours_valeur = 0
        else:
            jours_valeur = (solde / achat) * 30 if achat != 0 else 0
        
        jours.append(jours_valeur)
    
    # On ajoute les jours à la colonne 'Jours' du DataFrame
    data['Jours'] = jours
    # On sélectionne les colonnes 'Mois' et 'Jours'
    data = data[[index_column, "Jours"]]
    # On remplace les valeurs manquantes par 0
    data = data.fillna(0)
    # On arrondit les valeurs de la colonne 'Jours'
    data['Jours'] = data['Jours'].map(custom_round)

    
    return data

In [39]:
def credit_moyen_client(df):
    
    ventes = df[df['CompteNum'].astype(str).str[:3].astype(int).between(700, 709) & ~df['CompteNum'].astype(str).str.startswith('706')]
    prestations = df[df['CompteNum'].astype(str).str.startswith('706')]

    def generer_cmc(data,type:Literal["Ventes de marchandises","Prestation de services"]):

        data['Mois'] = pd.Categorical(data['Mois'], categories=mois_en_ordre, ordered=True)

        data[type] = data["Credit"]- data["Debit"]    
        cmc = data.groupby(['Mois'])[type].sum().reset_index()

        cmc[type] = cmc[type].map(custom_round)

        cmc.loc[len(cmc)] = ['Total', cmc[type].sum()]

        return cmc

    table_ventes = generer_cmc(  ventes,
                                 "Ventes de marchandises")
    table_prestations = generer_cmc(  prestations,
                                      "Prestation de services",)
    
    table_soldes,_ = generer_table_soldes(df,
                                        'Clients')
        

    table_cmc = pd.merge(table_prestations,table_ventes,on='Mois')

    table_cmc["Total TTC"] = ((table_cmc["Prestation de services"] + table_cmc["Ventes de marchandises"])*1.2).astype("int")
    table_cmc = table_cmc.merge(table_soldes,on = 'Mois')

    tables_jours = generer_table_jours(table_cmc,'Clients')
    table_cmc = table_cmc.merge(tables_jours,on = 'Mois',how='left')


    return table_cmc    

### Pourquoi y-a t'il N/A pour Décembre 2021

In [40]:
cmc = credit_moyen_client(df)
cmc

,Mois,Prestation de services,Ventes de marchandises,Total TTC,Soldes,Jours
0,Janvier,26454,0,31744,44288,0
1,Février,52037,0,62444,31430,15
2,Mars,98909,0,118690,81054,20
3,Avril,83273,0,99927,95015,29
4,Mai,39631,0,47557,53614,32
5,Juin,207775,0,249330,176376,21
6,Juillet,43127,0,51752,123859,39
7,Août,24800,0,29760,47452,40
8,Septembre,48220,0,57864,39176,20
9,Octobre,31566,0,37879,12879,10


In [41]:
def credit_moyen_fournisseur(df,previous_year_data=None):    
    """
    Génère un tableau pivot montrant l'évolution du crédit moyen fournisseurs pour chaque mois de l'année spécifiée, avec une ligne 'Total' à la fin.
    
    Args:
        df: DataFrame contenant les données de la balance comptable

    Returns:
        table_cmf: Un DataFrame contenant le crédit moyen fournisseurs mensuel
    """
    # Ajouter cette ligne au début de la fonction
    mois_en_ordre = reorder_months(df)
        
    achats_ttc = df[(df["JournalCode"].str.strip().isin(['AC','HA','A10']))]
    if achats_ttc.empty:
        achats_ttc = df[(df["JournalCode"].astype(str) == "60")]


    achats_ttc['Mois'] = pd.Categorical(achats_ttc['Mois'], categories=mois_en_ordre, ordered=True)

    achats_ttc['Achats TTC'] = achats_ttc["Debit"]    

    tables_achats = achats_ttc.groupby(['Mois'])['Achats TTC'].sum().reset_index()        

    tables_achats['Achats TTC'] = tables_achats['Achats TTC'].map(custom_round)
    


    table_soldes,total_solde = generer_table_soldes(df,
                                        type = 'Fournisseurs')
        
    table_cmf = pd.merge(tables_achats,table_soldes,on='Mois')

    tables_jours = generer_table_jours(table_cmf,'Fournisseurs',previous_year_data=previous_year_data)
    table_cmf = table_cmf.merge(tables_jours,on = 'Mois')
    
    # Add the Total row back at the end
    total_row = {'Mois': 'Total'}
    for col in table_cmf.columns:
        if col!='Mois':
            if col =='Achats TTC':
                total_row[col] = table_cmf[col].sum()
            elif col == 'Soldes':
                total_row[col] = total_solde
            else:
                total_row[col] = ""  # or some other appropriate value
    
    table_cmf = pd.concat([table_cmf, pd.DataFrame([total_row])], ignore_index=True)

    return table_cmf

In [42]:
cmf = credit_moyen_fournisseur(df)
cmf

,Mois,Achats TTC,Soldes,Jours
0,Janvier,54913,30847,17
1,Février,27723,34700,34
2,Mars,45127,42707,28
3,Avril,21337,18491,26
4,Mai,99038,93503,28
5,Juin,55253,40389,22
6,Juillet,32849,41834,35
7,Août,13583,15113,31
8,Septembre,25435,28951,38
9,Octobre,9011,4726,16


## Top fournisseur/ clients

In [43]:
def top_dix(df,type):
    """
    Génère un tableau des 10 principaux fournisseurs ou clients pour une année donnée.

    Parameters
    ----------
    df : pandas.DataFrame
        Le DataFrame contenant les données de la balance comptable.
    annee : int
        L'année pour laquelle le top 10 doit être calculé.
    type : Literal["Fournisseurs", "Clients"]
        Indique si le classement doit être fait pour les "Fournisseurs" ou les "Clients".
    show_title : bool, optional
        Si True, affiche le titre du tableau, par défaut False.

    Returns
    -------
    pd.DataFrame
        Un DataFrame contenant les 10 premiers fournisseurs ou clients, avec un résumé
        des autres et un total. Chaque ligne inclut le numéro et le libellé du compte auxiliaire,
        le solde et le pourcentage du total.
    """

    compte_num = ['401'] if type =="Fournisseurs" else ["411"]


    liste_journal_code_str = ['AC', 'VE', 'HA', 'VT','VTE','A10','ACH',"VEN"]
    liste_journal_code_int = ["60","70"]
    liste_journal_code = liste_journal_code_str + liste_journal_code_int

    def solde(data,type):
        return data["Credit"] - data["Debit"] if type =="Clients" else data["Debit"] - data["Credit"]

    def classement_top_10(data,compte_num_column,compte_lib_column):
            
        classement = data.groupby([compte_num_column,compte_lib_column])['Solde'].sum().reset_index().sort_values(by='Solde',ascending = False)        
        classement["Solde"] = classement["Solde"].map(custom_round)
        classement_top_10 = classement.head(10)
        def contrepartie(row):
            contrepartie_data = data.loc[data[compte_lib_column] == row[compte_lib_column]]
            ecriture_num_ids = list(contrepartie_data["EcritureNum"].value_counts().keys())
            new_df = df.loc[(df["EcritureNum"].isin(ecriture_num_ids)) & (df['JournalCode'].str.strip().isin(liste_journal_code))]
            data_without_compte_num = new_df.loc[~new_df[compte_num_column].astype(str).str[:3].isin(compte_num + ['44'])]
            data_without_compte_num["Solde"] = solde(data_without_compte_num,type)
            grouped_df = data_without_compte_num.groupby("CompteNum")["Solde"].sum().to_frame().reset_index()
            return grouped_df[grouped_df["Solde"] == grouped_df["Solde"].max()].reset_index()["CompteNum"][0]
        
        classement_top_10["Contrepartie"] = classement_top_10.apply(lambda row: contrepartie(row),axis=1)
        classement_top_10_tail = pd.DataFrame([{compte_num_column:"",
                                                compte_lib_column:f'Autres {type}',
                                                "Solde":classement.iloc[10:]["Solde"].sum(),
                                                "Contrepartie":""
                                                },
                                               {compte_num_column:"Total",
                                                compte_lib_column:'',
                                                "Solde":classement["Solde"].sum(),
                                                "Contrepartie":""
                                                }
                                            ])
        classement_top_10_final = pd.concat([classement_top_10,classement_top_10_tail],axis = 0)
        classement_top_10_final["Pourcentage"] = classement_top_10_final.apply(lambda row: round((row["Solde"]/classement_top_10_final["Solde"].iloc[-1])*100),axis = 1)
        classement_top_10_final.set_index(compte_num_column, inplace=True)
        classement_top_10_final = classement_top_10_final.reset_index()
        classement_top_10_final = classement_top_10_final[[compte_num_column,compte_lib_column,"Solde","Pourcentage","Contrepartie"]]

        return classement_top_10_final,compte_lib_column
    
    data = df[(df["JournalCode"].astype(str).str.strip().isin(liste_journal_code_str))].loc[df['CompteNum'].astype(str).str[:3].isin(compte_num)]

    if data.shape[0] == 0:        
        data = df[(df["JournalCode"].astype(str).str.strip().isin(liste_journal_code_int))].loc[df['CompteNum'].astype(str).str[:3].isin(compte_num)]
    
    if data.empty:
        return pd.DataFrame([{"CompteNum": "", "CompteLib": '', "Solde": ''}]), None
        
    if type =="Fournisseurs":
        data["Solde"] = data["Credit"] - data["Debit"]
    else:
        data["Solde"] = data["Debit"] - data["Credit"]

    try:
        classement_top_10_final,compte_lib_column = classement_top_10(data,compte_num_column = "CompAuxNum",compte_lib_column = "CompAuxLib")

    except ValueError:
        
        classement_top_10_final,compte_lib_column = classement_top_10(data,compte_num_column = "CompteNum",compte_lib_column = "CompteLib")

    return classement_top_10_final, compte_lib_column
    

In [44]:
type="Fournisseurs"
compte_num = ['401'] if type =="Fournisseurs" else ["411"]

liste_journal_code_str = ['AC', 'VE', 'HA', 'VT','VTE','A10','ACH',"VEN"]
liste_journal_code_int = ["60","70"]
data = df[(df["JournalCode"].astype(str).str.strip().isin(liste_journal_code_str))].loc[df['CompteNum'].astype(str).str[:3].isin(compte_num)]

if data.shape[0] == 0:        
    data = df[(df["JournalCode"].astype(str).str.strip().isin(liste_journal_code_int))].loc[df['CompteNum'].astype(str).str[:3].isin(compte_num)]



In [45]:
compte_lib_column = "CompAuxLib"
compte_num_column = "CompteNum"
contrepartie_data = data.loc[data[compte_lib_column] == "ROUX ARMANE "]
data_without_compte_num = contrepartie_data.loc[~(contrepartie_data[compte_num_column].astype(str).str[:3].isin(compte_num + ['44']))]
data_without_compte_num

,JournalCode,JournalLib,EcritureNum,EcritureDate,CompteNum,CompteLib,CompAuxNum,CompAuxLib,PieceRef,PieceDate,EcritureLib,Debit,Credit,EcritureLet,DateLet,ValidDate,Montantdevise,Idevise,Année,Mois


In [46]:
top_dix_fournisseurs,_ = top_dix(df,'Fournisseurs')
top_dix_fournisseurs

,CompAuxNum,CompAuxLib,Solde,Pourcentage,Contrepartie
0,0CAMBR,CAMBRIDGE ASSESMENT PS EXO,240494,60,60400000
1,0OLE,OLESUND,20978,5,60400000
2,0CCIPA,CCI PARIS EXO,16352,4,60400000
3,0ACE,ACE FINANCE PS,15032,4,62260000
4,0JFT,JFT GESTION PS,14127,4,61320000
5,0GLOBA,GLOBAL EXAM,10482,3,60400000
6,0ESP,ESPACE LA ROCHEFOUCAUD,8294,2,61320000
7,0BRIGH,BRIGHTLANGUAGE EXO,6265,2,60400000
8,0PEOP,PEOPLECERT,6180,2,60400000
9,0IHS,IHS53,5872,1,60400000


In [47]:
top_dix_clients ,_= top_dix(df,'Clients')
top_dix_clients


,CompteNum,CompteLib,Solde
0,,,


### Si future erreur pour pieceref tester ecriturenum

In [48]:
def top_dix_hors_taxes(df,type):

    compte_num,liste_journal_code_str,liste_journal_code_int = ('401',['AC','HA','A10'],["60"]) if type =="Fournisseurs" else ("411",['VE','VT','VTE'],["70"])
    compte_num_columns = ["CompAuxNum","CompteNum"]
    compte_lib_columns = ["CompAuxLib","CompteLib"]

    def next_value_after(target,list):
        """
        Returns the next value after the first occurrence of `target` in the list `lst`.
        Returns None if the target is not found or if it is the last element.
        """
        try:
            idx = list.index(target)
            if idx < len(list) - 1:
                return list[idx + 1]
        except ValueError:
            pass  # target not in list
        return None

    liste_journal_code = liste_journal_code_str
    if df.loc[(df['JournalCode'].str.strip().isin(liste_journal_code))].empty :
        liste_journal_code = liste_journal_code_int
    
    def solde(data,type):
        return data["Credit"] - data["Debit"] if type =="Fournisseurs" else data["Debit"] - data["Credit"]
    
    def classement_top_10(data,type,compte_num ,compte_num_column,compte_lib_column):
        
        def has_tva_top_10(df,type,row,compte_num,compte_num_column= compte_num_column,compte_lib_column=compte_lib_column):
            
            data = df.loc[df[compte_lib_column] == row[compte_lib_column]]
            ecriture_num_ids = list(data["EcritureNum"].value_counts().keys())
            new_df = df.loc[(df["EcritureNum"].isin(ecriture_num_ids)) & (df['JournalCode'].str.strip().isin(liste_journal_code)) ]
            data_without_compte_num = new_df.loc[~new_df[compte_num_column].astype(str).str.startswith((compte_num, '44'))]
            if new_df.equals(data_without_compte_num):
                data_without_compte_num = new_df.loc[~new_df[next_value_after(compte_num_column,compte_num_columns)].astype(str).str.startswith((compte_num, '44'))]

            data_without_compte_num["Solde"] = - solde(data_without_compte_num,type)
            grouped_df = data_without_compte_num.groupby("CompteNum")["Solde"].sum().to_frame().reset_index()
            # Find the row with the max value
            contrepartie = grouped_df[grouped_df["Solde"] == grouped_df["Solde"].max()].reset_index()["CompteNum"][0] 
            montant_ht = custom_round(data_without_compte_num["Solde"].sum())
            
            # Si les deux montants sont différents ca veut dire qu'il y'a une TVA appliquée
            return pd.Series({"Solde HT":montant_ht,
                              "Contrepartie":contrepartie})
        

        def has_tva_tail(df, tail_data, type, compte_num, compte_num_column=compte_num_column, compte_lib_column=compte_lib_column):
            # Get all EcritureLib values for the tail entries
            tail_ecritures = []
            for lib in tail_data[compte_lib_column].unique():
                comp_data = df.loc[df[compte_lib_column] == lib]
                tail_ecritures.extend(list(comp_data["EcritureNum"].value_counts().keys()))
            
            # Filter the dataframe based on these EcritureLib values
            new_df = df.loc[(df["EcritureNum"].isin(tail_ecritures)) & (df['JournalCode'].str.strip().isin(liste_journal_code))]
            data_without_compte_num = new_df.loc[~new_df[compte_num_column].astype(str).str.startswith((compte_num, '44'))]
            if new_df.equals(data_without_compte_num):
                data_without_compte_num = new_df.loc[~new_df[next_value_after(compte_num_column,compte_num_columns)].astype(str).str.startswith((compte_num, '44'))]
            
            data_without_compte_num["Solde"] = -solde(data_without_compte_num, type)
            montant_ht = custom_round(data_without_compte_num["Solde"].sum())
            
            return montant_ht

        classement = data.groupby([compte_num_column,compte_lib_column])['Solde'].sum().reset_index().sort_values(by='Solde',ascending = False)
                
        classement["Solde"] = classement["Solde"].map(custom_round)        

        classement_top_10 = classement.head(10)
        classement_top_10[["Solde HT","Contrepartie"]] = classement_top_10.apply(lambda row : has_tva_top_10(df,type,row,compte_num),axis = 1)
        
        tail_data = classement.iloc[10:]
        solde_ht_autres = has_tva_tail(df,tail_data,type=type,compte_num=compte_num)
        
        classement_top_10_tail = pd.DataFrame([{compte_num_column: "",compte_lib_column:f'Autres {type}',"Solde":classement.iloc[10:]["Solde"].sum(),"Solde HT":solde_ht_autres,"Contrepartie":""},
                                               {compte_num_column:"Total",compte_lib_column:'',"Solde":classement["Solde"].sum(),"Solde HT": classement_top_10["Solde HT"].sum() +solde_ht_autres ,"Contrepartie":""}])
        classement_top_10_final = pd.concat([classement_top_10,classement_top_10_tail],axis = 0)
        
        classement_top_10_final["Pourcentage HT"] = classement_top_10_final.apply(lambda row: round((row["Solde HT"]/classement_top_10_final["Solde HT"].iloc[-1])*100),axis = 1)
        classement_top_10_final.set_index(compte_num_column, inplace=True)
        classement_top_10_final = classement_top_10_final.reset_index()
        classement_top_10_final = classement_top_10_final[[compte_num_column,compte_lib_column,"Solde","Solde HT","Pourcentage HT","Contrepartie"]]

        return classement_top_10_final,compte_lib_column
    
    data = df[(df["JournalCode"].astype(str).str.strip().isin(liste_journal_code_str))].loc[df['CompteNum'].astype(str).str.startswith(compte_num)]
    
    if data.shape[0] == 0:        
        data = df[(df["JournalCode"].astype(str).str.strip().isin(liste_journal_code_int))].loc[df['CompteNum'].astype(str).str.startswith(compte_num)]
    
    if data.empty:
        return pd.DataFrame([{"CompteNum": "", "CompteLib": '', "Solde": ''}]), None
        
    data["Solde"] = solde(data,type)

    try:
        classement_top_10_final,compte_lib_column = classement_top_10(data,type,compte_num,compte_num_column = compte_num_columns[0],compte_lib_column = compte_lib_columns[0])
    except ValueError:
        classement_top_10_final,compte_lib_column = classement_top_10(data,type,compte_num,compte_num_column = compte_num_columns[1],compte_lib_column = compte_lib_columns[1])

    return classement_top_10_final, compte_lib_column
    

In [49]:
hors_taxes_top_dix_fournisseurs,_ = top_dix_hors_taxes(df,'Fournisseurs')
hors_taxes_top_dix_fournisseurs

,CompAuxNum,CompAuxLib,Solde,Solde HT,Pourcentage HT,Contrepartie
0,0CAMBR,CAMBRIDGE ASSESMENT PS EXO,240494,240494,62,60400000
1,0OLE,OLESUND,20978,20978,5,60400000
2,0CCIPA,CCI PARIS EXO,16352,16352,4,60400000
3,0ACE,ACE FINANCE PS,15032,12527,3,62260000
4,0JFT,JFT GESTION PS,14127,12233,3,61320000
5,0GLOBA,GLOBAL EXAM,10482,8839,2,60400000
6,0ESP,ESPACE LA ROCHEFOUCAUD,8294,6912,2,61320000
7,0BRIGH,BRIGHTLANGUAGE EXO,6265,6265,2,60400000
8,0PEOP,PEOPLECERT,6180,6180,2,60400000
9,0IHS,IHS53,5872,5702,1,60400000


In [50]:
# compte_lib_column = "CompteLib"
# compte_num_column = "CompAuxNum"
# type = "Fournisseurs"
# element = top_dix_fournisseurs.iloc[9]["CompteLib"]
# compte_num_columns = ["CompAuxNum","CompteNum"]
# compte_lib_columns = ["CompAuxLib","CompteLib"]

# def next_value_after(target,list):
#     """
#     Returns the next value after the first occurrence of `target` in the list `lst`.
#     Returns None if the target is not found or if it is the last element.
#     """
#     try:
#         idx = list.index(target)
#         if idx < len(list) - 1:
#             return list[idx + 1]
#     except ValueError:
#         pass  # target not in list
#     return None

# compte_num,liste_journal_code_str,liste_journal_code_int = ('401',['AC','HA','A10'],["60"]) if type =="Fournisseurs" else ("411",['VE','VT','VTE'],["70"])

# liste_journal_code = liste_journal_code_str

# if df.loc[(df['JournalCode'].str.strip().isin(liste_journal_code))].empty :
#     liste_journal_code = liste_journal_code_int
    
# def solde(data,type):
#     return data["Credit"] - data["Debit"] if type =="Fournisseurs" else data["Debit"] - data["Credit"]

# data = df.loc[df["CompAuxLib"] == element]
# piece_ref_ids = list(data["EcritureNum"].value_counts().keys())
# new_df = df.loc[(df["EcritureNum"].isin(piece_ref_ids)) & (df['JournalCode'].str.strip().isin(liste_journal_code)) ]
# data_without_compte_num = new_df.loc[~new_df[compte_num_column].astype(str).str.startswith((compte_num, '44'))]
# if new_df.equals(data_without_compte_num):
#     data_without_compte_num = new_df.loc[~new_df[next_value_after(compte_num_column,compte_num_columns)].astype(str).str.startswith((compte_num, '44'))]

# data_without_compte_num["Solde"] = - solde(data_without_compte_num,type)
# grouped_df = data_without_compte_num.groupby("CompteNum")["Solde"].sum().to_frame().reset_index()
# # # Find the row with the max value
# contrepartie = grouped_df[grouped_df["Solde"] == grouped_df["Solde"].max()].reset_index()["CompteNum"][0]
# montant_ht = custom_round(data_without_compte_num["Solde"].sum())
# montant_ht

In [51]:
hors_taxes_top_dix_clients,_ = top_dix_hors_taxes(df,'Clients')
hors_taxes_top_dix_clients

,CompteNum,CompteLib,Solde
0,,,


In [52]:
# compte_lib_column = "CompAuxLib"
# compte_num_column = "CompAuxNum"
# type = "Clients"
# element = top_dix_clients.iloc[0]["CompAuxLib"]
# compte_num_columns = ["CompAuxNum","CompteNum"]
# compte_lib_columns = ["CompAuxLib","CompteLib"]

# def next_value_after(target,list):
#     """
#     Returns the next value after the first occurrence of `target` in the list `lst`.
#     Returns None if the target is not found or if it is the last element.
#     """
#     try:
#         idx = list.index(target)
#         if idx < len(list) - 1:
#             return list[idx + 1]
#     except ValueError:
#         pass  # target not in list
#     return None

# compte_num,liste_journal_code_str,liste_journal_code_int = ('401',['AC','HA','A10'],["60"]) if type =="Fournisseurs" else ("411",['VE','VT'],["70"])

# liste_journal_code = liste_journal_code_str

# if df.loc[(df['JournalCode'].str.strip().isin(liste_journal_code))].empty :
#     liste_journal_code = liste_journal_code_int
    
# def solde(data,type):
#     return data["Credit"] - data["Debit"] if type =="Fournisseurs" else data["Debit"] - data["Credit"]

# data = df.loc[df["CompAuxLib"] == element]
# # ecriture_lib_ids = list(data["EcritureLib"].value_counts().keys())
# # new_df = df.loc[(df["EcritureLib"].isin(ecriture_lib_ids)) & (df['JournalCode'].str.strip().isin(liste_journal_code)) ]
# piece_ref_ids = list(data["PieceRef"].value_counts().keys())
# new_df = df.loc[(df["PieceRef"].isin(piece_ref_ids)) & (df['JournalCode'].str.strip().isin(liste_journal_code)) ]
# data_without_compte_num = new_df.loc[~new_df[compte_num_column].astype(str).str.startswith((compte_num, '44'))]
# if new_df.equals(data_without_compte_num):
#     data_without_compte_num = new_df.loc[~new_df[next_value_after(compte_num_column,compte_num_columns)].astype(str).str.startswith((compte_num, '44'))]

# data_without_compte_num["Solde"] = - solde(data_without_compte_num,type)
# montant_ht = custom_round(data_without_compte_num["Solde"].sum())
# montant_ht

## `####################################################`

### Fournisseur --> EcritureLib

In [53]:
# fournisseur = top_dix_fournisseurs.iloc[0]["CompAuxLib"]

# fournisseur_data = df.loc[df["CompAuxLib"] == fournisseur]
# ecriture_lib_ids = list(fournisseur_data["PieceRef"].value_counts().keys())
# new_df = df.loc[df["PieceRef"].isin(ecriture_lib_ids)]
# table_solde_data = fournisseur_data

# achats_ttc = new_df[(new_df["JournalCode"].str.strip().isin(['AC','HA','A10']))]
# if achats_ttc.empty:
#     achats_ttc = new_df[(new_df["JournalCode"].astype(str) == "60")]

# def generer_cmf(data):

#     data['Mois'] = pd.Categorical(data['Mois'], categories=mois_en_ordre, ordered=True)

#     data["Achats TTC"] = data["Debit"]    
#     cmf = data.groupby(['Mois'])["Achats TTC"].sum().reset_index()        

#     cmf["Achats TTC"] = cmf["Achats TTC"].map(custom_round)

#     cmf.loc[len(cmf)] = ['Total', cmf["Achats TTC"].sum()]

#     return cmf

# tables_achats = generer_cmf(achats_ttc)
# table_soldes,_ = generer_table_soldes(df,
#                                     'Fournisseurs',
#                                     table_solde_data = table_solde_data)

# table_cmf = pd.merge(tables_achats,table_soldes,on='Mois')

# tables_jours = generer_table_jours(table_cmf,'Fournisseurs')
# table_cmf = table_cmf.merge(tables_jours,on = 'Mois')
# print(fournisseur)
# table_cmf

### Client --> EcritureLib

In [54]:
# client = top_dix_clients.iloc[0]["CompAuxLib"]
# compte_num = '411'
# client_data = df.loc[df["CompAuxLib"] == client]
# ecriture_lib_ids = list(client_data["PieceRef"].value_counts().keys())
# new_df = df.loc[df["PieceRef"].isin(ecriture_lib_ids)]
# table_solde_data = new_df

# ventes = new_df[new_df['CompteNum'].astype(str).str[:3].astype(int).between(700, 709) & ~new_df['CompteNum'].astype(str).str.startswith('706')]
# prestations = new_df[new_df['CompteNum'].astype(str).str.startswith('706')]

# def generer_cmc(data,type:Literal["Ventes de marchandises","Prestation de services"]):

#     data['Mois'] = pd.Categorical(data['Mois'], categories=mois_en_ordre, ordered=True)

#     data[type] = data["Credit"] - data["Debit"]    
#     cmc = data.groupby(['Mois'])[type].sum().reset_index()

#     cmc[type] = cmc[type].map(custom_round)

#     cmc.loc[len(cmc)] = ['Total', cmc[type].sum()]

#     return cmc

# table_ventes = generer_cmc(  ventes,
#                                 "Ventes de marchandises")
# table_prestations = generer_cmc(  prestations,
#                                     "Prestation de services",)

# table_soldes,_ = generer_table_soldes(df,
#                                     'Clients',
#                                     table_solde_data=table_solde_data)    

# table_cmc = pd.merge(table_prestations,table_ventes,on='Mois')

# table_cmc["Total TTC"] = ((table_cmc["Prestation de services"] + table_cmc["Ventes de marchandises"])*1.2).astype("int")
# table_cmc = table_cmc.merge(table_soldes,on = 'Mois')

# tables_jours = generer_table_jours(table_cmc,'Clients')
# table_cmc = table_cmc.merge(tables_jours,on = 'Mois',how='left')

# print(client)
# table_cmc

## `####################################################`

##  Achats et sous traitances

In [55]:
def achats_et_sous_traitances(df):

    achats = df[df['CompteNum'].astype(str).str[:3].astype(int).between(601, 609)& ~df['CompteNum'].astype(str).str[:3].isin(['603','604','606','611'])]
    sous_traitance = df[df['CompteNum'].astype(str).str[:3].isin(['604','611'])]
    mois_en_ordre = reorder_months(df)
    
    def generer_table(data,type):
        data['Mois'] = pd.Categorical(data['Mois'], categories=mois_en_ordre, ordered=True)
        data[type] = data["Debit"] - data["Credit"]   
        data = data.groupby(['Mois'])[type].sum().reset_index()        

        data[type] = data[type].map(custom_round)

        data.loc[len(data)] = ['Total', data[type].sum()]

        return data
    
    table_achats = generer_table(achats,"Achats")
    table_sous_traitance = generer_table(sous_traitance,"Sous-traitance")

    table_achat_sous_traitance = pd.merge(table_achats,table_sous_traitance,on='Mois')
    table_achat_sous_traitance["Total"] = table_achat_sous_traitance["Achats"] + table_achat_sous_traitance["Sous-traitance"]

    table_achat_sous_traitance_cumule,_ = calculate_custom_cumsum(table_achat_sous_traitance[:-1])
    table_achat_sous_traitance_cumule = table_achat_sous_traitance_cumule[["Mois","Total"]]

    return table_achat_sous_traitance, table_achat_sous_traitance_cumule

In [56]:
table_achat_sous_traitance,table_achat_sous_traitance_cumule = achats_et_sous_traitances(df)
table_achat_sous_traitance

,Mois,Achats,Sous-traitance,Total
0,Janvier,0,39121,39121
1,Février,0,17157,17157
2,Mars,0,36489,36489
3,Avril,0,20701,20701
4,Mai,0,94345,94345
5,Juin,0,72934,72934
6,Juillet,0,46379,46379
7,Août,0,21243,21243
8,Septembre,0,21831,21831
9,Octobre,0,12178,12178


In [57]:
table_achat_sous_traitance_cumule

,Mois,Total
0,Janvier,39121
1,Février,56278
2,Mars,92767
3,Avril,113468
4,Mai,207813
5,Juin,280747
6,Juillet,327126
7,Août,348369
8,Septembre,370200
9,Octobre,382378


## Investissement

In [58]:
def investissement(df,annee):

    data = df[df['CompteNum'].astype(str).str[:2].astype(int).between(20, 26)]
    data_an = df[df['CompteNum'].astype(str).str[:2].astype(int).between(20, 26)].loc[df['JournalCode'].str.strip().isin(['AA','AN'])]

    colonne_annee = str(annee)

    # Calculer les montants nets (Débit - Crédit)
    data['Solde'] =  data["Debit"] - data["Credit"]
    data = data.groupby(['CompteNum','CompteLib'])['Solde'].sum().reset_index()
    data['Solde']= data['Solde'].map(custom_round)
    data.rename(columns={'Solde': colonne_annee}, inplace=True)
    
    data_an['Solde'] =  data_an["Debit"] - data_an["Credit"]
    data_an = data_an.groupby(['CompteNum','CompteLib'])['Solde'].sum().reset_index()
    data_an['Solde']= data_an['Solde'].map(custom_round)
    data_an.rename(columns={'Solde': colonne_annee}, inplace=True)

    data[colonne_annee] = data[colonne_annee] - data_an[colonne_annee]
    data.loc[len(data)] = ["","Total",data[colonne_annee].sum()]
    total_investissement = pd.DataFrame([{"Année":colonne_annee,"Total":data[colonne_annee][:-1].sum()}])
    
    return data ,total_investissement   

In [59]:
table_investissement,total_investissement = investissement(df,annee)
table_investissement

,CompteNum,CompteLib,2025
0,20110000,Frais de constitution,0
1,20510000,25/04/2017,0
2,20800000,Autres immobilisations incorpo,0
3,21820000,Materiel de transport,0
4,21830000,Materiel de bureau et informat,0
5,21831000,25/04/2017,0
6,23200000,Immob. incorporelles en cours,0
7,26110000,Actions,0
8,,Total,0


In [60]:
total_investissement

,Année,Total
0,2025,0


In [61]:
import os
path = "//ATC_FILE_SERVER/Documents Partagés/A classer/IT/Automates"
if os.path.exists(path):
    print("Le chemin existe.")
else:
    print("Nope")

Le chemin existe.
